# Streaming y SSE — Práctica

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/fdd_p26/blob/main/clase/18_intro_a_apis/code/02_streaming_sse.ipynb)


## 1. Setup

Instalamos `httpx`, una librería moderna de HTTP que soporta streaming de manera nativa.

In [ ]:
%pip install httpx matplotlib

In [ ]:
import httpx
import time
import json
import matplotlib.pyplot as plt

## 2. Request sin streaming

En una petición normal (buffered), el cliente espera a que el servidor
termine de enviar **toda** la respuesta antes de procesarla.

Usaremos `httpbin.org/drip` que envía datos gota a gota (simula un servidor lento).

In [ ]:
# Petición buffered (sin streaming)
# drip: envía 10 bytes, 1 cada 0.5 segundos (total ~5s)
url = "https://httpbin.org/drip?numbytes=10&duration=5&delay=0&code=200"

inicio = time.time()
respuesta = httpx.get(url, timeout=30)
fin = time.time()

print(f"Tamaño de respuesta: {len(respuesta.content)} bytes")
print(f"Tiempo total (buffered): {fin - inicio:.2f} segundos")
print(f"Primer byte recibido: hasta que terminó todo ({fin - inicio:.2f}s)")

## 3. Request con streaming

Con streaming, procesamos los datos **conforme llegan** del servidor.
Esto nos permite mostrar resultados parciales al usuario sin esperar la respuesta completa.

La métrica clave es el **TTFT** (Time To First Token): el tiempo que pasa
hasta que recibimos el primer fragmento de datos.

In [ ]:
# Petición con streaming
url = "https://httpbin.org/drip?numbytes=10&duration=5&delay=0&code=200"

inicio = time.time()
ttft = None
chunks_recibidos = []

with httpx.stream("GET", url, timeout=30) as respuesta:
    for chunk in respuesta.iter_bytes():
        ahora = time.time()
        if ttft is None:
            ttft = ahora - inicio
        chunks_recibidos.append({
            "tiempo": ahora - inicio,
            "bytes": len(chunk)
        })
        print(f"  Chunk recibido: {len(chunk)} bytes a los {ahora - inicio:.2f}s")

fin = time.time()
print(f"\nTTFT (Time To First Token): {ttft:.2f} segundos")
print(f"Tiempo total: {fin - inicio:.2f} segundos")
print(f"Chunks totales: {len(chunks_recibidos)}")

## 4. Parsing de eventos SSE

**Server-Sent Events (SSE)** es un protocolo donde el servidor envía eventos
en formato texto, cada uno precedido por `data: `. Así es como funcionan
las APIs de LLMs (OpenAI, Anthropic, etc.).

Usaremos `httpbin.org/stream/{n}` que envía `n` líneas JSON en streaming.

In [ ]:
# httpbin.org/stream/{n} envía n objetos JSON, uno por línea
url = "https://httpbin.org/stream/5"

with httpx.stream("GET", url, timeout=30) as respuesta:
    print("Content-Type:", respuesta.headers.get("content-type"))
    print()
    for linea in respuesta.iter_lines():
        if linea.strip():
            evento = json.loads(linea)
            print(f"  Evento #{evento['id']}: url={evento['url']}")

In [ ]:
# Simulación de parsing SSE como lo haría un cliente de LLM
# En una API real de LLM, cada línea viene como:
#   data: {"choices": [{"delta": {"content": "Hola"}}]}

def parsear_sse(lineas_raw):
    """Parsea líneas en formato SSE y extrae los datos."""
    eventos = []
    for linea in lineas_raw:
        linea = linea.strip()
        if linea.startswith("data: "):
            contenido = linea[6:]  # quitar "data: "
            if contenido == "[DONE]":
                break
            eventos.append(json.loads(contenido))
    return eventos

# Ejemplo con datos simulados (así se ven las respuestas de APIs de LLMs)
lineas_simuladas = [
    'data: {"id": 1, "texto": "Hola"}',
    'data: {"id": 2, "texto": " mundo"}',
    'data: {"id": 3, "texto": " desde"}',
    'data: {"id": 4, "texto": " ITAM"}',
    'data: [DONE]',
]

eventos = parsear_sse(lineas_simuladas)
texto_completo = "".join(e["texto"] for e in eventos)
print(f"Texto reconstruido: '{texto_completo}'")
print(f"Eventos parseados: {len(eventos)}")

## 5. Comparación de tiempos

Comparamos la experiencia del usuario entre modo buffered y streaming.
La diferencia clave es el **TTFT** (Time To First Token).

In [ ]:
url = "https://httpbin.org/drip?numbytes=10&duration=3&delay=0&code=200"

# --- Modo buffered ---
inicio = time.time()
respuesta = httpx.get(url, timeout=30)
tiempo_buffered = time.time() - inicio
# En modo buffered, TTFT = tiempo total (no vemos nada hasta que termina)
ttft_buffered = tiempo_buffered

# --- Modo streaming ---
inicio = time.time()
ttft_streaming = None
with httpx.stream("GET", url, timeout=30) as respuesta:
    for chunk in respuesta.iter_bytes():
        if ttft_streaming is None:
            ttft_streaming = time.time() - inicio
tiempo_streaming = time.time() - inicio

print("Comparación de tiempos:")
print(f"  Buffered  - TTFT: {ttft_buffered:.2f}s | Total: {tiempo_buffered:.2f}s")
print(f"  Streaming - TTFT: {ttft_streaming:.2f}s | Total: {tiempo_streaming:.2f}s")
print(f"\nEl streaming muestra el primer resultado {ttft_buffered/ttft_streaming:.1f}x más rápido")

## 6. Visualización

Gráfica comparativa de la latencia percibida entre ambos modos.

In [ ]:
# Gráfica de barras comparando TTFT y tiempo total
fig, ax = plt.subplots(figsize=(8, 5))

categorias = ["Buffered", "Streaming"]
ttft_valores = [ttft_buffered, ttft_streaming]
total_valores = [tiempo_buffered, tiempo_streaming]

x = range(len(categorias))
ancho = 0.35

barras_ttft = ax.bar(
    [i - ancho/2 for i in x], ttft_valores,
    ancho, label="TTFT (primer dato)", color="#e74c3c"
)
barras_total = ax.bar(
    [i + ancho/2 for i in x], total_valores,
    ancho, label="Tiempo total", color="#3498db"
)

ax.set_ylabel("Tiempo (segundos)")
ax.set_title("Latencia percibida: Buffered vs Streaming")
ax.set_xticks(x)
ax.set_xticklabels(categorias)
ax.legend()

# Agregar valores sobre las barras
for barra in barras_ttft:
    ax.text(barra.get_x() + barra.get_width()/2., barra.get_height() + 0.05,
            f"{barra.get_height():.2f}s", ha="center", va="bottom", fontsize=10)
for barra in barras_total:
    ax.text(barra.get_x() + barra.get_width()/2., barra.get_height() + 0.05,
            f"{barra.get_height():.2f}s", ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()

### Nota sobre APIs de LLMs

Las APIs de modelos de lenguaje (OpenAI, Anthropic, etc.) usan exactamente este
patrón de streaming con SSE. La diferencia práctica es:

- El `Content-Type` es `text/event-stream`
- Cada chunk contiene un token o fragmento de texto generado
- La respuesta termina con `data: [DONE]`

El código de parsing que vimos en la sección 4 es esencialmente lo que hacen
las librerías cliente como `openai` o `anthropic` internamente.